In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from pathlib import Path

import sim_function

figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)


In [ ]:
# For Figure 2, run once per theta (10, 1, 0.1). The three PDFs were assembled in Keynote.
# Base parameter set
para = {
    'nubar': 1,
    'theta': 10,
    'delta': 0, 
    'boundary': 1000,
    'ngen': 128,
    'ninit': 1
}


# Simulation parameters
sidelength = 128
reps = 100

In [ ]:
# Simulation
deltaseq = np.linspace(0, 0.3, sidelength)
nubarseq = np.linspace(0.5, 1.5, sidelength)
nubar_grid, delta_grid = np.meshgrid(nubarseq, deltaseq)
nubar_flat = nubar_grid.flatten()
delta_flat = delta_grid.flatten()
n = len(nubar_flat)
boundary_hit = np.full(n, 0)
final_size = np.full(n, 0)
final_size_subcritical = np.full(n, 0)

start_time = time.time()
for i in range(n):
  
  # timer
  elapsed_time = time.time() - start_time
  avg_time_per_iteration = elapsed_time / (i + 1)
  remaining_time = (n - i - 1) * avg_time_per_iteration
  remaining_minutes = remaining_time / 60
  
  print(
    f'Location {i+1} of {n} - Elapsed time: {elapsed_time:.2f} sec - Estimated time remaining: {remaining_minutes:.2f} minutes',
    end='\r', flush=True
  )
  
  # main loop
  for j in range(reps):
    
    # simulate
    para['nubar'] = nubar_flat[i]
    para['delta'] = delta_flat[i]
    out = sim_function.sim(para)
    
    # write results
    boundary_hit[i] += int(out['boundary_hit'])
    final_size[i] += out['final_size']
    if not out['boundary_hit']:
      final_size_subcritical[i] += out['final_size']


In [ ]:
# Prepare for plotting
x = nubarseq
y = deltaseq
fssc = final_size_subcritical.reshape(sidelength, sidelength)
fs = final_size.reshape(sidelength, sidelength)
bh = boundary_hit.reshape(sidelength, sidelength)

if reps > 1:
  fssc = fssc / (reps - bh)
  fs = fs / reps
  bh = bh / reps

In [ ]:
# Plotting

fontsize = 20

fig, axes = plt.subplots(2, 1, figsize=(5, 8), sharex=True, sharey=True)

xticks = [0.7, 1.0, 1.3]
yticks = [0.0, 0.1, 0.2, 0.3]

for ax in axes:
    ax.tick_params(labelbottom=True)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

# --- First subplot: fssc ---
norm1 = Normalize(vmin=5, vmax=30)
ax = axes[0]
im1 = ax.imshow(
    fssc,
    origin='lower',
    extent=[x.min(), x.max(), y.min(), y.max()],
    aspect='auto',
    cmap='hot',
    norm=norm1
)

ax.set_ylabel(r'$\delta$', fontsize=fontsize)

# Overlay curve
k = para['theta']
xx = np.linspace(0.5, 1.5, 100000)
deltastar = (1 - xx) / (1 - xx + 1 / k)
deltastar[(deltastar < 0) | (deltastar > y.max())] = np.nan
ax.plot(xx, deltastar, color='white', linewidth=2)

# Add colorbar for this subplot
cbar1 = fig.colorbar(im1, ax=ax, orientation='vertical', fraction=0.05, pad=0.04)
cbar1.set_label("Final Size", fontsize=0.8*fontsize)
cbar1.ax.tick_params(labelsize=0.8*fontsize)

# --- Second subplot: bh ---
norm2 = Normalize(vmin=0, vmax=0.5)
ax = axes[1]
im2 = ax.imshow(
    bh,
    origin='lower',
    extent=[x.min(), x.max(), y.min(), y.max()],
    aspect='auto',
    cmap='bone',
    norm=norm2
)

ax.set_xlabel(r'$\bar{\nu}$', fontsize=fontsize)
ax.set_ylabel(r'$\delta$', fontsize=fontsize)

# Overlay the same curve
ax.plot(xx, deltastar, color='white', linewidth=2)

# Add colorbar for this subplot
cbar2 = fig.colorbar(im2, ax=ax, orientation='vertical', fraction=0.05, pad=0.04)
cbar2.set_label("BH Value", fontsize=0.8*fontsize)
cbar2.ax.tick_params(labelsize=0.8*fontsize)

# Apply tick font sizes for both axes
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=0.8*fontsize)

plt.tight_layout()

theta = para['theta']
filename = f"heatmaps_k_{theta}.pdf"
outpath = figures_dir / filename
fig.savefig(outpath, bbox_inches="tight")

plt.show()



In [ ]:
outpath